# Statistical Demand Forecasting Baselines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmavigo/retail-demand-forecasting/blob/main/notebooks/01_m5_demand_forecasting.ipynb)

**Business question:** How many units should each store expect to sell during the next 28 days?

## 1. Environment and official data

In [ ]:
import sys, subprocess
from pathlib import Path

if 'google.colab' in sys.modules:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'kagglehub>=1.0,<2', 'pandas>=2.2,<3', 'numpy>=2,<3',
        'plotly>=5.24,<7', 'scikit-learn>=1.5,<2', 'lightgbm>=4.5,<5'
    ])

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
pd.set_option('display.max_columns', 30)


In [ ]:
import kagglehub

# Download latest version. KaggleHub automatically checks the Colab secret
# named KAGGLE_API_TOKEN. If it is missing, the login widget opens once.
try:
    path = kagglehub.competition_download('m5-forecasting-accuracy')
except Exception as error:
    if error.__class__.__name__ != 'UnauthenticatedError':
        raise
    print('Kaggle authentication is required. Paste your Kaggle API token in the login form below.')
    kagglehub.login()
    path = kagglehub.competition_download('m5-forecasting-accuracy')

print("Path to competition files:", path)

DATA_DIR = Path(path)
if not (DATA_DIR / 'calendar.csv').exists():
    DATA_DIR = next(p.parent for p in DATA_DIR.rglob('calendar.csv'))

required = {'calendar.csv', 'sell_prices.csv', 'sales_train_evaluation.csv'}
available = {p.name for p in DATA_DIR.glob('*.csv')}
assert required.issubset(available), f"Missing files: {sorted(required - available)}"


## 2. Load the demand matrix and calendar

In [ ]:
sales = pd.read_csv(DATA_DIR / 'sales_train_evaluation.csv')
calendar = pd.read_csv(DATA_DIR / 'calendar.csv', parse_dates=['date'])
day_cols = [c for c in sales if c.startswith('d_')]
calendar_days = calendar.set_index('d').loc[day_cols].reset_index()
values = sales[day_cols].to_numpy(dtype=np.float32)

quality = pd.Series({
    'series': len(sales), 'days': len(day_cols), 'products': sales.item_id.nunique(),
    'stores': sales.store_id.nunique(), 'duplicate_ids': sales.id.duplicated().sum(),
    'negative_sales': int((values < 0).sum()), 'zero_share': float((values == 0).mean()),
})
quality.to_frame('value')


## 3. Preserve the final 28 days as unseen data
Time-series validation must respect time. Random train/test splitting would allow future information to leak into training.

In [ ]:
def score(actual, predicted, history):
    error = actual - predicted
    scale = np.mean(np.diff(history, axis=1) ** 2, axis=1)
    usable = scale > 0
    denominator = np.abs(actual).sum()
    return {
        'MAE': float(np.abs(error).mean()),
        'WAPE': float(np.abs(error).sum() / denominator),
        'RMSSE': float(np.sqrt(np.mean(error[usable] ** 2, axis=1) / scale[usable]).mean()),
        'Bias': float(error.sum() / denominator),
    }


In [ ]:
HORIZON = 28
TRAIN_END = values.shape[1] - HORIZON
train, actual = values[:, :TRAIN_END], values[:, TRAIN_END:]

forecasts = {
    'Last value': np.repeat(train[:, -1:], HORIZON, axis=1),
    'Mean of last 7 days': np.repeat(train[:, -7:].mean(axis=1, keepdims=True), HORIZON, axis=1),
    'Mean of last 28 days': np.repeat(train[:, -28:].mean(axis=1, keepdims=True), HORIZON, axis=1),
    'Seasonal lag 7': np.tile(train[:, -7:], (1, 4)),
    'Seasonal lag 28': train[:, -28:].copy(),
}
results = pd.DataFrame([{'model': n, **score(actual, p, train)} for n, p in forecasts.items()]).sort_values('WAPE')
display(results.style.format({'MAE': '{:.4f}', 'WAPE': '{:.2%}', 'RMSSE': '{:.4f}', 'Bias': '{:.2%}'}))


## 4. Compare aggregate demand patterns

In [ ]:
best_name = results.iloc[0].model
best = forecasts[best_name]
dates = calendar_days.loc[TRAIN_END:, 'date'].reset_index(drop=True)
forecast_daily = pd.DataFrame({'date': dates, 'actual': actual.sum(axis=0), 'forecast': best.sum(axis=0)})
px.line(forecast_daily, x='date', y=['actual', 'forecast'], markers=True, title=f'Holdout demand: actual versus {best_name}').show()
px.bar(results.sort_values('WAPE', ascending=False), x='WAPE', y='model', orientation='h', text_auto='.1%', title='Forecast accuracy comparison').show()


## 5. Diagnose performance by store

In [ ]:
store_rows = []
for store in sorted(sales.store_id.unique()):
    mask = sales.store_id.eq(store).to_numpy()
    store_rows.append({
        'store_id': store,
        'actual_units': float(actual[mask].sum()),
        'forecast_units': float(best[mask].sum()),
        'WAPE': float(np.abs(actual[mask] - best[mask]).sum() / actual[mask].sum()),
    })
store_results = pd.DataFrame(store_rows).sort_values('WAPE')
display(store_results.style.format({'actual_units': '{:,.0f}', 'forecast_units': '{:,.0f}', 'WAPE': '{:.1%}'}))
px.bar(store_results, x='store_id', y='WAPE', text_auto='.1%', title='Forecast error by store').show()


## Conclusion
The strongest simple method becomes the benchmark that every machine-learning model must beat on the same untouched 28-day horizon.